In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import pandas as pd
import matplotlib.pyplot as plt

# Carga de datos
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Procesamiento de df_train
# LLENAR DATOS VACIOS
df_train.loc[df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].isnull(), 'ESTU_VALORMATRICULAUNIVERSIDAD'] = 'No pago matricula'
df_train.loc[df_train['ESTU_HORASSEMANATRABAJA'].isnull(), 'ESTU_HORASSEMANATRABAJA'] = '0'
df_train.loc[df_train['ESTU_PAGOMATRICULAPROPIO'].isnull(), 'ESTU_PAGOMATRICULAPROPIO'] = 'No'
df_train.loc[df_train['FAMI_ESTRATOVIVIENDA'].isnull(), 'FAMI_ESTRATOVIVIENDA'] = 'Sin Estrato'

# Corregir caracteres especiales
df_train['ESTU_PRGM_ACADEMICO'] = df_train['ESTU_PRGM_ACADEMICO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_PRGM_DEPARTAMENTO'] = df_train['ESTU_PRGM_DEPARTAMENTO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_VALORMATRICULAUNIVERSIDAD'] = df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_HORASSEMANATRABAJA'] = df_train['ESTU_HORASSEMANATRABAJA'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


df_train.drop(columns=['FAMI_TIENEINTERNET.1'], inplace=True)

# Eliminar columnas que no sirven
df_train.drop(columns=['FAMI_TIENELAVADORA', 'FAMI_TIENEAUTOMOVIL', 'ESTU_PRIVADO_LIBERTAD', 'FAMI_EDUCACIONMADRE', 'FAMI_EDUCACIONPADRE'], inplace=True)


df_train['frecuencia_ESTU_VALORMATRICULAUNIVERSIDAD'] = df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].map(df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].value_counts())
df_train.drop(columns=['ESTU_VALORMATRICULAUNIVERSIDAD'], inplace=True)

df_train['frecuencia_ESTU_PRGM_DEPARTAMENTO'] = df_train['ESTU_PRGM_DEPARTAMENTO'].map(df_train['ESTU_PRGM_DEPARTAMENTO'].value_counts())
df_train.drop(columns=['ESTU_PRGM_DEPARTAMENTO'], inplace=True)

df_train['frecuencia_ESTU_HORASSEMANATRABAJA'] = df_train['ESTU_HORASSEMANATRABAJA'].map(df_train['ESTU_HORASSEMANATRABAJA'].value_counts())
df_train.drop(columns=['ESTU_HORASSEMANATRABAJA'], inplace=True)

df_train['frecuencia_ESTU_PRGM_ACADEMICO'] = df_train['ESTU_PRGM_ACADEMICO'].map(df_train['ESTU_PRGM_ACADEMICO'].value_counts())
df_train.drop(columns=['ESTU_PRGM_ACADEMICO'], inplace=True)


# Mapeos
mapeo_estrato = {
    'Sin Estrato': 0,
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6
}
mapeo_si_no = {
    'No': 0,
    'Si': 1,
}
df_train['FAMI_ESTRATOVIVIENDA'] = df_train['FAMI_ESTRATOVIVIENDA'].map(mapeo_estrato)
df_train['FAMI_TIENEINTERNET'] = df_train['FAMI_TIENEINTERNET'].map(mapeo_si_no)
df_train['ESTU_PAGOMATRICULAPROPIO'] = df_train['ESTU_PAGOMATRICULAPROPIO'].map(mapeo_si_no)
df_train['FAMI_TIENECOMPUTADOR'] = df_train['FAMI_TIENECOMPUTADOR'].map(mapeo_si_no)


# Convertir RENDIMIENTO_GLOBAL a categórico
dummies = df_train['RENDIMIENTO_GLOBAL'].str.get_dummies()
df_train = pd.concat([df_train, dummies], axis=1)
df_train.drop(columns=['RENDIMIENTO_GLOBAL'], inplace=True)

df_train.to_csv('train_modified.csv', index=False)



# Procesamiento de df_test
df_test.loc[df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].isnull(), 'ESTU_VALORMATRICULAUNIVERSIDAD'] = 'No pago matricula'
df_test.loc[df_test['ESTU_HORASSEMANATRABAJA'].isnull(), 'ESTU_HORASSEMANATRABAJA'] = '0'
df_test.loc[df_test['ESTU_PAGOMATRICULAPROPIO'].isnull(), 'ESTU_PAGOMATRICULAPROPIO'] = 'No'
df_test.loc[df_test['FAMI_ESTRATOVIVIENDA'].isnull(), 'FAMI_ESTRATOVIVIENDA'] = 'Sin Estrato'

df_test['ESTU_PRGM_ACADEMICO'] = df_test['ESTU_PRGM_ACADEMICO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_PRGM_DEPARTAMENTO'] = df_test['ESTU_PRGM_DEPARTAMENTO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_VALORMATRICULAUNIVERSIDAD'] = df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_HORASSEMANATRABAJA'] = df_test['ESTU_HORASSEMANATRABAJA'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


df_test.drop(columns=['FAMI_TIENEINTERNET.1'], inplace=True)

df_test.drop(columns=['FAMI_TIENELAVADORA', 'FAMI_TIENEAUTOMOVIL', 'ESTU_PRIVADO_LIBERTAD', 'FAMI_EDUCACIONMADRE', 'FAMI_EDUCACIONPADRE'], inplace=True)

df_test['frecuencia_ESTU_VALORMATRICULAUNIVERSIDAD'] = df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].map(df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].value_counts())
df_test.drop(columns=['ESTU_VALORMATRICULAUNIVERSIDAD'], inplace=True)

df_test['frecuencia_ESTU_PRGM_DEPARTAMENTO'] = df_test['ESTU_PRGM_DEPARTAMENTO'].map(df_test['ESTU_PRGM_DEPARTAMENTO'].value_counts())
df_test.drop(columns=['ESTU_PRGM_DEPARTAMENTO'], inplace=True)

df_test['frecuencia_ESTU_HORASSEMANATRABAJA'] = df_test['ESTU_HORASSEMANATRABAJA'].map(df_test['ESTU_HORASSEMANATRABAJA'].value_counts())
df_test.drop(columns=['ESTU_HORASSEMANATRABAJA'], inplace=True)

df_test['frecuencia_ESTU_PRGM_ACADEMICO'] = df_test['ESTU_PRGM_ACADEMICO'].map(df_test['ESTU_PRGM_ACADEMICO'].value_counts())
df_test.drop(columns=['ESTU_PRGM_ACADEMICO'], inplace=True)



df_test['FAMI_ESTRATOVIVIENDA'] = df_test['FAMI_ESTRATOVIVIENDA'].map(mapeo_estrato)
df_test['FAMI_TIENEINTERNET'] = df_test['FAMI_TIENEINTERNET'].map(mapeo_si_no)
df_test['ESTU_PAGOMATRICULAPROPIO'] = df_test['ESTU_PAGOMATRICULAPROPIO'].map(mapeo_si_no)
df_test['FAMI_TIENECOMPUTADOR'] = df_test['FAMI_TIENECOMPUTADOR'].map(mapeo_si_no)


df_test.to_csv('test_modified.csv', index=False)







In [3]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
import pandas as pd
import matplotlib.pyplot as plt

# Carga de datos
df_train = pd.read_csv('train_modified.csv')
df_test = pd.read_csv('test_modified.csv')

# Suponiendo que las columnas de rendimiento son 'alto', 'bajo', 'medio-alto', 'medio-bajo'
y_train = df_train[['alto', 'bajo', 'medio-alto', 'medio-bajo']]
X_train = df_train.drop(columns=['alto', 'bajo', 'medio-alto', 'medio-bajo'])

# Necesitamos una columna con la etiqueta en forma de string
y_train_labels = y_train.idxmax(axis=1)

imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)

# Entrenar el modelo de K-Nearest Neighbors
model = KNeighborsClassifier(n_neighbors=40)
model.fit(X_train_imputed, y_train_labels)

# Carga de IDs del conjunto de prueba original
df_test_ids = pd.read_csv('test.csv')[['ID']]  # Suponiendo que 'ID' es la columna de identificación

# Imputar valores faltantes en df_test
X_test_imputed = imputer.transform(df_test)

# Verificar la consistencia del tamaño del conjunto de prueba después del preprocesamiento
if len(X_test_imputed) != len(df_test_ids):
    raise ValueError("The length of the test set does not match the length of the test IDs.")

# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test_imputed)

# Verificar nuevamente que las longitudes coincidan
if len(y_pred) != len(df_test_ids):
    raise ValueError("The number of predictions does not match the number of test IDs.")

# Generar el archivo CSV con las predicciones
df_results = pd.DataFrame({'ID': df_test_ids['ID'], 'RENDIMIENTO_GLOBAL': y_pred})
df_results.to_csv('predicciones_knn.csv', index=False)

